# Notebook 4: Mastering Tools — The Agent's Hands

**What you'll learn:** function_tool decorator, Pydantic tools, async tools, multi-tool agents, agent-as-tool pattern, and real-world agentic tool workflows.

---

## Why Tools Are The Heart of Agentic AI

Without tools, an agent is just a chatbot:
```
LLM alone:  "I think the weather might be sunny" (hallucination)
LLM + Tool: calls get_weather("Lahore") → "32°C, humid" (real data)
```

Tools transform LLMs from **text generators** into **action takers**.

### The Agentic Tool Loop:
```
User: "What's the weather in Lahore and should I carry an umbrella?"
  │
  ▼
Agent THINKS: "I need weather data. Let me call get_weather."
  │
  ▼
Agent ACTS: get_weather(city="Lahore") → {temp: 32, humidity: 85, rain: true}
  │
  ▼
Agent OBSERVES: "It's 32°C with 85% humidity and rain expected."
  │
  ▼
Agent RESPONDS: "It's 32°C in Lahore with rain expected. Yes, carry an umbrella!"
```

In [ ]:
# uv add openai-agents openai ollama

In [ ]:
# Setup for local Ollama (or remove model= to use OpenAI)
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled


set_tracing_disabled(True)

def get_ollama_model(model_name: str = "llama3.1:8b"):
    client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    return OpenAIChatCompletionsModel(model=model_name, openai_client=client)

# Switch between local and cloud:
MODEL = get_ollama_model()        # Local (free)
# MODEL = "gpt-4.1-mini"          # Cloud (paid, uncomment to use)

print("Model ready")

---

## Tool Type 1: @function_tool Decorator (Simplest)

Turn ANY Python function into an agent tool with one decorator.

In [ ]:
from agents import Agent, Runner, function_tool


@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # In production: call a real weather API
    weather_data = {
        "lahore": "32°C, Humid, Rain expected",
        "karachi": "28°C, Sunny, Clear skies",
        "islamabad": "22°C, Cloudy, Light drizzle",
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")


@function_tool
def get_time(timezone: str) -> str:
    """Get the current time in a timezone."""
    from datetime import datetime
    return f"Current time: {datetime.now().strftime('%H:%M:%S')} (local)"


# Agent with multiple tools
assistant = Agent(
    name="City Assistant",
    instructions="""
    You help users with city information.
    Use the get_weather tool when asked about weather.
    Use the get_time tool when asked about time.
    Always call the appropriate tool — never guess!
    """,
    model=MODEL,
    tools=[get_weather, get_time]  # <-- Pass tools as a list
)

result = await Runner.run(assistant, "What's the weather like in Lahore right now?")
print(result.final_output)

### How @function_tool Works Under the Hood:

```python
@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""  # ← Becomes tool description
    ...

# The SDK automatically generates this JSON schema:
# {
#   "name": "get_weather",
#   "description": "Get the current weather for a city.",
#   "parameters": {
#     "type": "object",
#     "properties": {
#       "city": {"type": "string"}
#     },
#     "required": ["city"]
#   }
# }
```

The LLM sees this schema, decides when to call it, and the SDK handles execution automatically.

---

## Tool Type 2: Pydantic-Validated Tools (Production-Grade)

For complex inputs, use Pydantic models for **automatic validation**.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from agents import Agent, Runner, function_tool


class OrderRequest(BaseModel):
    """Schema for placing an order."""
    product_name: str = Field(description="Name of the product")
    quantity: int = Field(description="Number of items", ge=1, le=100)
    priority: Literal["standard", "express", "overnight"] = Field(
        description="Shipping priority"
    )


@function_tool
def place_order(order: OrderRequest) -> str:
    """Place an order for a product with specified quantity and shipping priority."""
    prices = {"standard": 0, "express": 500, "overnight": 1500}
    shipping_cost = prices[order.priority]
    return (
        f"Order placed!\n"
        f"Product: {order.product_name}\n"
        f"Quantity: {order.quantity}\n"
        f"Shipping: {order.priority} (+Rs.{shipping_cost})\n"
        f"Order ID: ORD-{hash(order.product_name) % 10000:04d}"
    )


@function_tool
def check_inventory(product_name: str) -> str:
    """Check if a product is available in inventory."""
    inventory = {
        "laptop": 15, "keyboard": 50, "mouse": 30, "monitor": 8
    }
    stock = inventory.get(product_name.lower(), 0)
    return f"{product_name}: {'In stock' if stock > 0 else 'Out of stock'} ({stock} units)"


shop_agent = Agent(
    name="Shop Assistant",
    instructions="""
    You are an e-commerce assistant.
    
    WORKFLOW:
    1. When customer wants to buy something, FIRST check_inventory
    2. If in stock, ask for quantity and shipping preference
    3. Then place_order with their choices
    4. If out of stock, suggest alternatives
    
    Always check inventory BEFORE placing an order.
    """,
    model=MODEL,
    tools=[check_inventory, place_order]
)

result = await Runner.run(
    shop_agent, 
    "I want to buy 2 keyboards with express shipping"
)
print(result.final_output)

### Notice the Agentic Behavior:
The agent doesn't just call one tool — it:
1. **Thinks** → "I should check inventory first"
2. **Acts** → Calls `check_inventory("keyboard")`
3. **Observes** → "50 units in stock"
4. **Thinks** → "Great, now I can place the order"
5. **Acts** → Calls `place_order({product: "keyboard", qty: 2, priority: "express"})`
6. **Responds** → Confirms the order to the user

This **multi-step reasoning** is what makes it an agent, not a chatbot.

---

## Tool Type 3: Async Tools (For Real APIs)

Production tools often call external APIs. Use async to avoid blocking.

In [ ]:
import asyncio
from agents import Agent, Runner, function_tool


@function_tool
async def search_database(query: str) -> str:
    """Search the company database for information."""
    # Simulate async database call
    await asyncio.sleep(0.5)  # Simulating network latency
    
    # Mock database
    records = {
        "revenue": "Q1 2025 Revenue: $2.3M (↑ 15% YoY)",
        "employees": "Total employees: 142 (Engineering: 68, Sales: 34, Ops: 40)",
        "customers": "Active customers: 1,247 (Enterprise: 89, SMB: 1,158)",
    }
    
    for key, value in records.items():
        if key in query.lower():
            return value
    return "No records found matching your query."


@function_tool
async def send_notification(recipient: str, message: str) -> str:
    """Send a notification to a team member."""
    await asyncio.sleep(0.3)  # Simulating API call
    return f"Notification sent to {recipient}: '{message}'"


operations_agent = Agent(
    name="Ops Agent",
    instructions="""
    You are an operations agent for a tech company.
    Use search_database to look up company data.
    Use send_notification to alert team members.
    
    When asked about metrics, always search the database first.
    If something is urgent, notify the relevant person.
    """,
    model=MODEL,
    tools=[search_database, send_notification]
)

result = await Runner.run(
    operations_agent, 
    "What's our current revenue? If it's above $2M, notify the CEO that we hit target."
)
print(result.final_output)

---

## Real-World Pattern: DevOps Incident Response Agent

**Industry Scenario:** A production monitoring agent that checks service health, queries logs, and pages the on-call engineer.

In [ ]:
import random
from datetime import datetime
from agents import Agent, Runner, function_tool


@function_tool
def check_service_health(service_name: str) -> str:
    """Check the health status of a microservice."""
    services = {
        "api-gateway":   {"status": "healthy",   "latency_ms": 45,  "error_rate": 0.1},
        "user-service":  {"status": "degraded",  "latency_ms": 850, "error_rate": 12.5},
        "payment-service":{"status": "healthy",   "latency_ms": 120, "error_rate": 0.3},
        "auth-service":  {"status": "healthy",   "latency_ms": 30,  "error_rate": 0.0},
    }
    data = services.get(service_name.lower(), None)
    if not data:
        return f"Service '{service_name}' not found. Available: {list(services.keys())}"
    return (
        f"Service: {service_name}\n"
        f"Status: {data['status']}\n"
        f"Latency: {data['latency_ms']}ms\n"
        f"Error Rate: {data['error_rate']}%"
    )


@function_tool
def query_logs(service_name: str, level: str) -> str:
    """Query recent logs for a service. Level can be: info, warn, error."""
    logs = [
        f"[ERROR] {service_name}: Connection pool exhausted - max connections reached",
        f"[ERROR] {service_name}: Timeout after 5000ms waiting for database response",
        f"[WARN]  {service_name}: Memory usage at 89% - approaching threshold",
        f"[ERROR] {service_name}: Failed to process request - retrying (attempt 3/3)",
    ]
    return f"Recent {level.upper()} logs for {service_name}:\n" + "\n".join(logs)


@function_tool
def page_oncall(engineer: str, severity: str, message: str) -> str:
    """Page the on-call engineer. Severity: P1, P2, P3."""
    return f"PAGED {engineer} [{severity}]: {message} (at {datetime.now().strftime('%H:%M')})"


@function_tool
def restart_service(service_name: str) -> str:
    """Restart a microservice. Use with caution."""
    return f"Service '{service_name}' restart initiated. ETA: 30 seconds."


incident_agent = Agent(
    name="Incident Response Agent",
    instructions="""
    You are a DevOps incident response agent.
    
    INCIDENT RESPONSE PROTOCOL:
    1. When alerted about an issue, FIRST check_service_health of the affected service
    2. If the service is degraded/down, query_logs with level="error" to understand why
    3. Based on severity:
       - Error rate > 10%: Page on-call as P1, attempt restart
       - Error rate 5-10%: Page on-call as P2
       - Error rate < 5%: Log and monitor (no page)
    4. Provide a clear incident summary
    
    On-call engineer: Ahmed (ahmed@company.com)
    
    Be methodical. Follow the protocol step by step.
    """,
    model=MODEL,
    tools=[check_service_health, query_logs, page_oncall, restart_service]
)

result = await Runner.run(
    incident_agent,
    "ALERT: user-service is showing high error rates. Users are reporting login failures."
)
print(result.final_output)

### What Just Happened (The Agentic Part):

The agent autonomously:
1. Checked health → found "degraded" status, 12.5% error rate
2. Queried logs → found connection pool and timeout errors
3. Determined severity → 12.5% > 10% → P1 incident
4. Paged the on-call engineer
5. Initiated a service restart
6. Provided a summary

**6 tool calls, all decided by the LLM. That's agentic AI.**

---

## Advanced: Agent as a Tool (Nested Agents)

An agent can USE another agent as a tool. This is the foundation of **multi-agent workflows**.

In [ ]:
from agents import Agent, Runner, function_tool

# Specialist agent: only does translation
translator = Agent(
    name="Translator",
    instructions="You translate text to Urdu. Return ONLY the translated text.",
    model=MODEL
)

# Specialist agent: only summarizes
summarizer = Agent(
    name="Summarizer",
    instructions="You summarize text in 1-2 sentences. Be concise.",
    model=MODEL
)

# Orchestrator agent: uses specialists as tools
orchestrator = Agent(
    name="Content Processor",
    instructions="""
    You process content using specialized agents.
    - Use translate_to_urdu when user wants translation
    - Use summarize_text when user wants a summary
    - You can chain them: summarize first, then translate the summary
    """,
    model=MODEL,
    tools=[
        translator.as_tool(
            tool_name="translate_to_urdu",
            tool_description="Translate any text to Urdu language"
        ),
        summarizer.as_tool(
            tool_name="summarize_text",
            tool_description="Summarize long text into 1-2 sentences"
        ),
    ]
)

result = await Runner.run(
    orchestrator,
    "Summarize and then translate to Urdu: 'Python is a high-level programming language "
    "known for its simplicity and readability. It is widely used in web development, "
    "data science, artificial intelligence, and automation.'"
)
print(result.final_output)

---

## Tool Design Best Practices

| Practice | Why |
|---|---|
| **Clear docstrings** | The LLM reads them to decide when to call the tool |
| **Type hints always** | SDK generates JSON schema from types |
| **Return strings** | The LLM processes the result as text |
| **Handle errors** | Return error messages, don't raise exceptions |
| **Single responsibility** | One tool = one action |
| **Descriptive names** | `check_inventory` > `do_stuff` |
| **Pydantic for complex inputs** | Validation + clear schema for the LLM |

```python
# BAD: Vague name, no docstring, no types
@function_tool
def do_it(x):
    return process(x)

# GOOD: Clear name, docstring, types, error handling
@function_tool
def search_products(query: str, category: str = "all", max_results: int = 5) -> str:
    """Search the product catalog by query. Optionally filter by category."""
    try:
        results = catalog.search(query, category, max_results)
        return json.dumps(results)
    except Exception as e:
        return f"Search failed: {str(e)}"
```

---

## Summary

| Tool Type | When to Use | Code |
|---|---|---|
| `@function_tool` | Simple functions | `@function_tool def f(x: str) -> str:` |
| Pydantic tool | Complex validated input | `def f(data: MyModel) -> str:` |
| Async tool | External API calls | `async def f(x: str) -> str:` |
| Agent as tool | Nested specialist agents | `agent.as_tool(name, description)` |

**Next:** See `agent.py` for a complete, production-ready agent combining ALL concepts!